In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from baseline.generate import generate
from evaluation.evaluator import (
    extract_code, evaluate, get_defined_functions, get_called_functions, 
    get_function_name, run_candidate, deduplicate_candidates,
    extract_call_args, get_probe_inputs, find_disagreements
)

Loading tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded!


In [2]:


def generate_candidates(problem_text, test_example, n=8, temperature=0.8):
    """Generate N diverse candidate solutions for one problem."""
    candidates = []
    for i in range(n):
        response = generate(
            problem_text,
            test_example=test_example,
            temperature=temperature,
            do_sample=True
        )
        code = extract_code(response)
        candidates.append(code)
    return candidates

In [3]:
def analyze_ambiguity(sample, n_candidates=8, temperature=0.8):
    problem_text = sample["text"]
    test_list = sample["test_list"]
    expected_func_name = get_called_functions(test_list)
    expected_func_name = list(expected_func_name)[0] if expected_func_name else None

    if expected_func_name is None:
        return {"error": "could not determine expected function name"}
    
    candidates = generate_candidates(
        problem_text,
        test_example=test_list[0],
        n=n_candidates,
        temperature=temperature
    )

    probe_inputs = get_probe_inputs(test_list)

    if not probe_inputs:
        return {
            "task_id": sample.get("task_id"),
            "note": "no simple-literal probe inputs available (custom class args?)",
            "n_valid_candidates": len(candidates),
        }
    unique_candidates = deduplicate_candidates(
        candidates, probe_inputs, expected_func_name
    )
    disagreements = find_disagreements(
        unique_candidates, probe_inputs, expected_func_name
    )

    return {
        "task_id": sample.get("task_id"),
        "n_raw_candidates": len(candidates),
        "n_unique_candidates": len(unique_candidates),
        "n_disagreement_points": len(disagreements),
        "disagreements": disagreements,
        "candidates": unique_candidates
    }

In [4]:
from datasets import load_dataset
dataset = load_dataset("google-research-datasets/mbpp")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'huggingface.co'. (_ssl.c:992)' thrown while requesting HEAD https://huggingface.co/datasets/google-research-datasets/mbpp/resolve/4bb6404fdc6cacfda99d4ac4205087b89d32030c/mbpp.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since google-research-datasets/mbpp couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'full' at C:\Users\sanim\.cache\huggingface\datasets\google-research-datasets___mbpp\full\0.0.0\4bb6404fdc6cacfda99d4ac4205087b89d32030c (last modified on Sat Jul 18 20:52:13 2026).


In [5]:
import json

results_phase2 = []
for i in range(10):  # start small — 10 problems, since each does 8 generations
    sample = dataset["train"][i]
    print(f"Analyzing task {sample.get('task_id', i)}...")
    result = analyze_ambiguity(sample, n_candidates=8, temperature=0.8)
    results_phase2.append(result)

with open("../results/phase2_ambiguity_analysis.json", "w") as f:
    json.dump(results_phase2, f, indent=2)

# Quick summary
for r in results_phase2:
    if "n_disagreement_points" in r:
        print(f"Task {r['task_id']}: {r['n_unique_candidates']} unique candidates, "
              f"{r['n_disagreement_points']} disagreement points")

Analyzing task 601...
Analyzing task 602...
Analyzing task 603...
[1, 5, 7]
[1, 5, 7]
[1, 5, 7]
[1, 5, 7]
[1, 5, 7]
[1, 5, 7]
Analyzing task 604...
Analyzing task 605...
Analyzing task 606...
Analyzing task 607...
Analyzing task 608...
Analyzing task 609...
Analyzing task 610...
Task 602: 2 unique candidates, 1 disagreement points
Task 603: 5 unique candidates, 3 disagreement points
Task 604: 1 unique candidates, 0 disagreement points
Task 605: 1 unique candidates, 0 disagreement points
Task 606: 2 unique candidates, 3 disagreement points
Task 607: 1 unique candidates, 0 disagreement points
Task 608: 3 unique candidates, 3 disagreement points
Task 609: 2 unique candidates, 2 disagreement points
Task 610: 4 unique candidates, 3 disagreement points


In [6]:
for r in results_phase2:
    if r.get("task_id") == 608:
        for d in r["disagreements"]:
            print(d)

{'input': (2,), 'output_groups': {'1': [0], '2': [2]}}
{'input': (3,), 'output_groups': {'3': [0], '4': [2]}}
{'input': (4,), 'output_groups': {'6': [0], '8': [2]}}


In [9]:
import json

results_phase2 = []
N = 30

for i in range(N):
    sample = dataset["train"][i]
    print(f"Analyzing task {sample.get('task_id', i)}...")
    result = analyze_ambiguity(sample, n_candidates=8, temperature=0.8)
    results_phase2.append(result)

    # save after every task, not just at the end
    with open("../results/phase2_ambiguity_analysis.json", "w") as f:
        json.dump(results_phase2, f, indent=2)

print("Done.")

Analyzing task 601...
Analyzing task 602...
Analyzing task 603...
Analyzing task 604...
Analyzing task 605...
Analyzing task 606...
Analyzing task 607...
Analyzing task 608...
Analyzing task 609...
Analyzing task 610...
Analyzing task 611...
Analyzing task 612...
Analyzing task 613...
Analyzing task 614...
Analyzing task 615...
Analyzing task 616...
Analyzing task 617...
Analyzing task 618...
Analyzing task 619...
Analyzing task 620...
1
1
1
3
3
3
4
4
4
1
3
4
1
3
4
1
3
4
Analyzing task 621...
['MSM', '240', 'is', '104', '129', 'best', '10']
['130', 'abc', '463']
['MSM', '240', 'is', '104', '129', 'best', '10']
['130', 'abc', '463']
['MSM', '240', 'is', '104', '129', 'best', '10']
['130', 'abc', '463']
['MSM', '240', 'is', '104', '129', 'best', '10']
['130', 'abc', '463']
['MSM', '240', 'is', '104', '129', 'best', '10']
['130', 'abc', '463']
['MSM', '240', 'is', '104', '129', 'best', '10']
['130', 'abc', '463']
Analyzing task 622...
Analyzing task 623...
Analyzing task 624...
Analyzing 

In [10]:
total = len([r for r in results_phase2 if "n_disagreement_points" in r])
ambiguous = len([r for r in results_phase2 if r.get("n_disagreement_points", 0) > 0])
no_probes = len([r for r in results_phase2 if "note" in r])

print(f"Total analyzable tasks: {total}")
print(f"Tasks with genuine disagreement: {ambiguous} ({ambiguous/total*100:.1f}%)")
print(f"Tasks skipped (no simple-literal probe inputs): {no_probes}")

Total analyzable tasks: 29
Tasks with genuine disagreement: 19 (65.5%)
Tasks skipped (no simple-literal probe inputs): 1


<h3>Known limitation:</h3> Early runs in this dataset show inflated disagreement counts for outputs that are map objects or custom-class instances without __repr__ defined (e.g. task 618, 623, 630) — these produce unique memory-address reprs every call regardless of actual value equality. A fix normalizing map/generator outputs to lists was applied to evaluator.py after this run to fix this issue. Tasks 601, 617, 630 involving custom classes (e.g. Pair) remain a known false-positive source for disagreement counts and should be interpreted cautiously.